In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Data Science with PySpark Connect and Distributed XGBoost

## Overview

In this notebook, you will learn how to parallelize and scale your data science and machine learning workflows on large datasets using distributed computing with Apache Spark (`PySpark`) using **Spark Connect**.

You will build a machine learning pipeline to predict NYC taxi tip amounts. You will focus on scaling data processing (`PySpark DataFrames`/`Spark SQL`) and training an `XGBoost` model via the Spark Connect-compatible Estimator API.

### Objectives

* Initialize a cluster-ready PySpark session using Spark Connect.
* Construct an end-to-end data pipeline with Spark DataFrames.
* Configure and train distributed XGBoost on Spark using the Connect-compatible Estimator API.
* Evaluate validation accuracy on a held-out test dataset.

## Setup

To get started, we need to initialize a `SparkSession` using Spark Connect.

### Install Dependencies

First, we will install the necessary packages for PySpark, XGBoost, and Spark Connect.

In [ ]:
# Install dependencies for Spark XGBoost and Spark Connect
!pip3 install pyarrow pandas xgboost==3.2.0 google-spark-connect==0.5.6 google-cloud-dataproc

#### Initialize Spark Session with Spark Connect

Initialize the `SparkSession` using Spark Connect, which allows you to connect to a remote Spark cluster.

In [ ]:
from google.cloud import dataproc_v1
from google.cloud.dataproc_v1 import Session, SparkConnectConfig
from google.cloud.spark_connect import GoogleSparkSession

project_id = "your-project-id"
location = "us-central1"

session_config = Session()
spark = GoogleSparkSession.builder.projectId(project_id).location(location).googleSessionConfig(session_config).getOrCreate()

---

## Prepare the NYC taxi dataset

This tutorial uses the [NYC Taxi & Limousine Commission (TLC) Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).

The dataset contains individual trip records from yellow taxis in New York City, and includes fields like:
*   Pick-up and drop-off dates, times, and locations
*   Trip distances
*   Itemized fare amounts
*   Passenger counts
*   **Tip amounts** (*the target variable we will predict*)

#### Define Data Path

Specify the Google Cloud Storage (GCS) path where the NYC taxi dataset is located.

In [ ]:
# Just using 1-month data of the NYC Taxi dataset
DATA_DIR = "gs://dataproc-metastore-public-binaries/nyc_taxi_data"
print(f"Using dataset from {DATA_DIR}")

### Load and Clean Data

Let's load the data we downloaded and clean out invalid trips.

In [ ]:
import time
from pyspark.sql.functions import col

start_time = time.perf_counter()

# 1. Load data
df = spark.read.parquet(f"{DATA_DIR}/*.parquet")
print(f"Loaded {df.count():,} records.")

# 2. Clean data
df = df.filter(
    (col('fare_amount') > 0) & (col('fare_amount') < 500) &
    (col('trip_distance') > 0) & (col('trip_distance') < 100) &
    (col('tip_amount') >= 0) & (col('tip_amount') < 100) &
    (col('payment_type') == 1) # Credit card only (tips recorded)
)
print(f"Clean records remaining: {df.count():,}.")
print(f"Load and Clean Time: {time.perf_counter() - start_time:.2f} seconds")

### Feature Engineering

Next, we prepare our features for the machine learning models. We will extract time-based features, log-transform the fare, and create route-specific aggregations based on the pickup and dropoff location IDs.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

start_time = time.perf_counter()

# Time Features
df = df.withColumn('hour', F.hour('tpep_pickup_datetime'))
df = df.withColumn('dow', F.dayofweek('tpep_pickup_datetime')) # Note: 1=Sunday, 7=Saturday in PySpark
df = df.withColumn('is_weekend', F.when((col('dow') == 1) | (col('dow') == 7), 1).otherwise(0))
df = df.withColumn('is_rush_hour', 
                   F.when(((col('hour') >= 7) & (col('hour') <= 9)) | 
                          ((col('hour') >= 17) & (col('hour') <= 19)), 1).otherwise(0))

# Amount Features
df = df.withColumn('fare_log', F.log1p('fare_amount'))
df = df.withColumn('fare_decimal', (col('fare_amount') % 1 * 100).cast('int'))
df = df.withColumn('is_round_fare', F.when(col('fare_amount') % 5 == 0, 1).otherwise(0))

# Route Features
df = df.withColumn('route_id', F.concat_ws('_', col('PULocationID').cast('string'), col('DOLocationID').cast('string')))
route_window = Window.partitionBy('route_id')
df = df.withColumn('route_frequency', F.count('*').over(route_window))

# Location Aggregation Features (Mean and Std Tip by Pickup Location)
pu_window = Window.partitionBy('PULocationID')
df = df.withColumn('pu_tip_mean', F.mean('tip_amount').over(pu_window))
df = df.withColumn('pu_tip_std', F.stddev('tip_amount').over(pu_window))
df = df.fillna({'pu_tip_std': 0.0})

#### Display Engineered Features

Display the first few rows of the DataFrame with the newly engineered features to inspect the results.

In [ ]:
df.show()

### Prepare Features and Target

Let's select our feature columns and split the data into a Train and Test set.

In [ ]:
from pyspark.sql import functions as F

feature_cols = [
    'trip_distance', 'fare_amount', 'passenger_count',
    'hour', 'dow', 'is_weekend', 'is_rush_hour',
    'fare_log', 'fare_decimal', 'is_round_fare',
    'route_frequency', 'pu_tip_mean', 'pu_tip_std',
    'PULocationID', 'DOLocationID'
]

# Calculate the mean for each feature column
mean_exprs = [F.mean(F.col(c)).alias(c) for c in feature_cols]
means_row = df.select(*mean_exprs).first()

# Convert the Row to a dictionary for fillna
means_dict = means_row.asDict()

# Guard against columns that might be completely null (which would return None)
clean_means_dict = {k: (v if v is not None else 0.0) for k, v in means_dict.items()}

# Impute the nulls
df_imputed = df.fillna(clean_means_dict)

# Replicate VectorAssembler
array_cols = [F.col(c).cast("double") for c in feature_cols]

# Create the 'features' column as an Array of Doubles
df_assembled = df_imputed.withColumn("features", F.array(*array_cols))

# Split Data
train_df, test_df = df_assembled.randomSplit([0.8, 0.2], seed=42)

print(f"Train rows: {train_df.count():,}")
print(f"Test rows: {test_df.count():,}")

---

## Train Models

Now we will train predictive models to forecast tip amounts using PySpark Connect and Distributed XGBoost.

### Distributed XGBoost
We perform distributed training using the `SparkXGBRegressor`. This performs high-performance distributed training across the cluster workers while being fully compatible with the Spark Connect thin-client model.

In [ ]:
from xgboost.spark import SparkXGBRegressor

start_time = time.perf_counter()

xgb = SparkXGBRegressor(
  features_col="features",
  label_col="tip_amount",
  num_workers=2,
  max_depth=5,
  learning_rate=0.1
)

xgb_model = xgb.fit(train_df)

xgb_preds = xgb_model.transform(test_df)
xgb_preds = xgb_preds.withColumnRenamed('prediction', 'xgb_prediction')

evaluator_xgb = RegressionEvaluator(labelCol="tip_amount", predictionCol="xgb_prediction", metricName="rmse")
xgb_rmse = evaluator_xgb.evaluate(xgb_preds)

print(f"\n{'XGBoost RMSE:':<20} ${xgb_rmse:.4f}")

---

## Evaluate the Model

Finally, let's review the performance of our XGBoost model.

### Model Performance

Review the RMSE for the trained Distributed XGBoost model to evaluate its predictive performance on the test dataset.

In [ ]:
from pyspark.sql.functions import col, lit

print(f"\n{'Model':<20} {'RMSE':>10}")
print("-" * 32)
print(f"{'XGBoost':<20} ${xgb_rmse:>9.4f}")
print("-" * 32)

### Stop Spark Session

Stop the Spark session to release the resources.

In [ ]:
spark.stop()